# Prepare Dataset

In [30]:
import json, string
import numpy as np

In [31]:
with open("data/alfred/train.json", "r") as file:
    train_dataset = json.load(file)

with open("data/alfred/valid_unseen.json", "r") as file:
    test_dataset_unseen = json.load(file)

with open("data/alfred/valid_seen.json", "r") as file:
    test_dataset_seen = json.load(file)

with open("data/stopwords.json", "r") as file:
    stopwords = set(json.load(file))


def text_to_words(text):
    return text.strip(string.punctuation + string.whitespace).lower().split()

In [32]:
from scipy import sparse

# Prepare train data
train_texts = [item["task"] for item in train_dataset]
train_labels = [item["label"] for item in train_dataset]
labels = list(set(train_labels))
label_index = {label: i for i, label in enumerate(labels)}
row_indices = [label_index[l] for l in train_labels]
col_indices = range(len(train_labels))
indicator_mat = sparse.csr_matrix(
    (np.ones(len(train_labels)), (row_indices, col_indices)),
    shape=(len(labels), len(train_labels)),
)

# validation (seen)
test_texts_seen = [item["task"] for item in test_dataset_seen]
test_labels_seen = [item["label"] for item in test_dataset_seen]

# validation (unseen)
test_texts_unseen = [item["task"] for item in test_dataset_unseen]
test_labels_unseen = [item["label"] for item in test_dataset_unseen]

# Method 1: Bag of Words (BoW)

For each label, consider all the task descriptions as its document content.
In BoW model, we simply count the occurences of each word, without considering the order.

In [33]:
from collections import defaultdict, Counter

# Construct the BoW for each label
BoW = defaultdict(Counter)
for item in train_dataset:
    BoW[item["label"]].update(text_to_words(item["task"]))

# Print the labels
for label in BoW.keys():
    print(label)

look_at_obj_in_light
pick_and_place_simple
pick_and_place_with_movable_recep
pick_clean_then_place_in_recep
pick_cool_then_place_in_recep
pick_heat_then_place_in_recep
pick_two_obj_and_place


In [34]:
def evaluate(pred_func, dataset, **kwargs):
    correct = 0
    for item in dataset:
        label = item["label"]
        predict = pred_func(item["task"], **kwargs)
        if label == predict:
            correct += 1
    print(f"Accuracy: {correct / len(dataset)}")

## 1.1 BoW Probability

First we consider predicting by the probability of words in each labels. That is:

$$
P(\text{text}|\text{label}) 
= \prod_{\text{word}\in\text{text}} p(\text{label}|\text{word})
= \prod_{\text{word}\in\text{text}} \frac{\text{tf}_{\text{word},\text{label}}}{\text{tf}_{\text{word}}}
$$

Obviously for calculation simplicity we can simply compare:

$$
\begin{aligned}
&\log P(\text{text}|\text{label}) \\
=& \sum_{\text{word}\in\text{text}} \left( \log{\text{tf}_{\text{word},\text{label}}} -\log{\text{tf}_{\text{word}}} \right) \\
=& \sum_{\text{word}\in\text{text}} \log{\text{tf}_{\text{word},\text{label}}}\ 
\underbrace{-\sum_{\text{word}\in\text{text}} \log{\text{tf}_{\text{word}}}}_\text{common part}
\end{aligned}
$$

In [35]:
def bow_prob(text, remove_stopwords=False):
    words = text_to_words(text)
    label_log_probs = {}
    for label, counter in BoW.items():
        log_prob = 0.0
        for word in words:
            if remove_stopwords and word in stopwords:
                continue
            log_prob += np.log(counter[word] + 1)
        label_log_probs[label] = log_prob
    return max(label_log_probs, key=label_log_probs.get)

In [36]:
evaluate(bow_prob, test_dataset_unseen, remove_stopwords=False)
evaluate(bow_prob, test_dataset_unseen, remove_stopwords=True)
evaluate(bow_prob, test_dataset_seen, remove_stopwords=False)
evaluate(bow_prob, test_dataset_seen, remove_stopwords=True)

Accuracy: 0.904993909866017
Accuracy: 0.8940316686967114
Accuracy: 0.8451219512195122
Accuracy: 0.8573170731707317


## 1.2 Vector Similarity

In [37]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score


def evaluate(texts, y_labels, remove_stopwords=False):
    vectorizer = CountVectorizer(
        stop_words=list(stopwords) if remove_stopwords else None
    )
    train_tf = vectorizer.fit_transform(train_texts)
    label_tf = normalize(indicator_mat @ train_tf, norm="l2", axis=1)
    test_tf = normalize(vectorizer.transform(texts), norm="l2", axis=1)
    similarity = test_tf @ label_tf.T
    predicted_indices = np.argmax(similarity, axis=1).A1
    predicted_labels = [labels[i] for i in predicted_indices]
    print(accuracy_score(y_labels, predicted_labels))

In [38]:
evaluate(test_texts_unseen, test_labels_unseen, remove_stopwords=False)
evaluate(test_texts_unseen, test_labels_unseen, remove_stopwords=True)
evaluate(test_texts_seen, test_labels_seen, remove_stopwords=False)
evaluate(test_texts_seen, test_labels_seen, remove_stopwords=True)

0.784409257003654
0.8380024360535931
0.7097560975609756
0.7560975609756098


## Methods 2

**TF-IDF with Supervised Classifiers**

To represent the instruction text numerically, we adopt the Term Frequency–Inverse Document Frequency (TF-IDF) representation. 
TF-IDF reflects the importance of a word in a document relative to the entire corpus.

For a term $t$ in document $d$, the TF-IDF weight is defined as

\begin{equation}
\text{TF-IDF}(t,d) = TF(t,d) \times IDF(t)
\end{equation}

where the term frequency (TF) is defined as

\begin{equation}
TF(t,d) = \frac{f_{t,d}}{\sum_{t'} f_{t',d}}
\end{equation}

Here, $f_{t,d}$ denotes the number of occurrences of term $t$ in document $d$.

The inverse document frequency (IDF) is defined as

\begin{equation}
IDF(t) = \log \left(\frac{N}{df(t)}\right)
\end{equation}

where $N$ is the total number of documents in the corpus and $df(t)$ is the number of documents containing term $t$.

Each document is therefore represented as a TF-IDF feature vector

\begin{equation}
\mathbf{x}_d = [w_1, w_2, \dots, w_V]
\end{equation}

where $V$ denotes the vocabulary size and $w_i$ represents the TF-IDF weight of term $i$.

The resulting TF-IDF feature vectors are used as input to several supervised classifiers, including Logistic Regression, Random Forest, Support Vector Machines (SVM), and k-Nearest Neighbors (KNN), in order to predict the task label from the instruction text.

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words=list(stopwords), max_features=1000, ngram_range=(1, 1)
)

X_train = vectorizer.fit_transform(train_texts)

X_test_seen = vectorizer.transform(test_texts_seen)
X_test_unseen = vectorizer.transform(test_texts_unseen)

print("Train shape:", X_train.shape)
print("Seen shape:", X_test_seen.shape)
print("Unseen shape:", X_test_unseen.shape)

Train shape: (21025, 1000)
Seen shape: (820, 1000)
Unseen shape: (821, 1000)


In [43]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score

log_model = LogisticRegression(max_iter=2000)

log_model.fit(X_train, train_labels)

pred_seen = log_model.predict(X_test_seen)
pred_unseen = log_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("Logistic Regression")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)

rf_model.fit(X_train, train_labels)

pred_seen = rf_model.predict(X_test_seen)
pred_unseen = rf_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("Random Forest")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

svm_model = LinearSVC()

svm_model.fit(X_train, train_labels)

pred_seen = svm_model.predict(X_test_seen)
pred_unseen = svm_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("SVM")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train, train_labels)

pred_seen = knn_model.predict(X_test_seen)
pred_unseen = knn_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("KNN")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

Logistic Regression
Seen Accuracy: 0.9158536585365854
Unseen Accuracy: 0.9439707673568819

Random Forest
Seen Accuracy: 0.9134146341463415
Unseen Accuracy: 0.928136419001218

SVM
Seen Accuracy: 0.9182926829268293
Unseen Accuracy: 0.9464068209500609

KNN
Seen Accuracy: 0.85
Unseen Accuracy: 0.8830694275274056



In [46]:
results = {}


def evaluate(model, name):
    model.fit(X_train, train_labels)

    pred_seen = model.predict(X_test_seen)
    pred_unseen = model.predict(X_test_unseen)

    acc_seen = accuracy_score(test_labels_seen, pred_seen)
    acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

    results[name] = (acc_seen, acc_unseen)


evaluate(LogisticRegression(max_iter=2000), "Logistic")
evaluate(RandomForestClassifier(n_estimators=200), "RandomForest")
evaluate(LinearSVC(), "SVM")
evaluate(KNeighborsClassifier(n_neighbors=5), "KNN")

print("Model Comparison")
for k, v in results.items():
    print(k, "Seen:", round(v[0], 4), "Unseen:", round(v[1], 4))

Model Comparison
Logistic Seen: 0.9256 Unseen: 0.9379
RandomForest Seen: 0.9232 Unseen: 0.9342
SVM Seen: 0.9244 Unseen: 0.9513
KNN Seen: 0.8768 Unseen: 0.8916


## Method 3

**Sentence Embedding with Pretrained Transformer**

To capture semantic information beyond word frequency features, we utilize a pretrained sentence embedding model to encode each task instruction into a dense vector representation.

Given an instruction text $d$, we obtain its embedding using a pretrained sentence encoder:

$$
\mathbf{x}_d = f(d)
$$

where $f(\cdot)$ denotes the pretrained sentence embedding model. In this work, we use the **all-MiniLM-L6-v2** model from Sentence-Transformers, which maps each instruction to a dense vector:

$$
\mathbf{x}_d \in \mathbb{R}^{384}
$$

The resulting embedding vector is then fed into a Multi-Layer Perceptron (MLP) classifier to predict the task label.

The hidden layer representation is computed as

$$
\mathbf{h} = \sigma(W_1 \mathbf{x}_d + b_1)
$$

where $W_1$ and $b_1$ are learnable parameters and $\sigma(\cdot)$ denotes the ReLU activation function.

The final output layer produces the probability distribution over task labels using a softmax function:

$$
\mathbf{y} = \text{softmax}(W_2 \mathbf{h} + b_2)
$$

where $\mathbf{y} \in \mathbb{R}^{C}$ and $C$ is the number of task classes.

In [11]:
# !pip install sentence-transformers

In [12]:
from sentence_transformers import SentenceTransformer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import numpy as np

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")

X_train_embed = model.encode(train_texts, show_progress_bar=True)

X_test_seen_embed = model.encode(test_texts_seen, show_progress_bar=True)

X_test_unseen_embed = model.encode(test_texts_unseen, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/658 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

In [47]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128), activation="relu", max_iter=500, random_state=42
)

mlp_model.fit(X_train_embed, train_labels)

NameError: name 'X_train_embed' is not defined

In [15]:
pred_seen = mlp_model.predict(X_test_seen_embed)

acc_seen = accuracy_score(test_labels_seen, pred_seen)

print("Sentence Embedding + MLP")
print("Seen Accuracy:", acc_seen)

Sentence Embedding + MLP
Seen Accuracy: 0.9365853658536586


In [16]:
pred_unseen = mlp_model.predict(X_test_unseen_embed)

acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("Unseen Accuracy:", acc_unseen)

Unseen Accuracy: 0.9269183922046285


## Method 4

### Method 4: SBERT with Cosine Similarity Classification

In this method, we classify task instructions using semantic similarity in the embedding space. Each instruction is first encoded into a dense vector using a pretrained sentence encoder.

Given an instruction text $d$, we compute its embedding as

$$
\mathbf{x}_d = f(d)
$$

where $f(\cdot)$ denotes the pretrained sentence embedding model (Sentence-BERT). In this work, we use the **all-MiniLM-L6-v2** model, which maps each instruction into a vector

$$
\mathbf{x}_d \in \mathbb{R}^{384}.
$$

For each task class $c$, we compute a prototype embedding by averaging the embeddings of all training samples belonging to that class:

$$
\boldsymbol{\mu}_c = \frac{1}{N_c} \sum_{i=1}^{N_c} \mathbf{x}_i
$$

where $N_c$ is the number of training samples with label $c$.

Given a new instruction embedding $\mathbf{x}$, we compute the cosine similarity between $\mathbf{x}$ and each class prototype:

$$
s_c = \cos(\mathbf{x}, \boldsymbol{\mu}_c)
= \frac{\mathbf{x} \cdot \boldsymbol{\mu}_c}{\|\mathbf{x}\| \|\boldsymbol{\mu}_c\|}
$$

The predicted label is the class with the highest similarity:

$$
\hat{y} = \arg\max_c s_c
$$

This method performs classification directly in the semantic embedding space without training an additional classifier.

In [17]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [18]:
# Collect embeddings by label
label_embeddings = {}

for label in set(train_labels):
    indices = [i for i, l in enumerate(train_labels) if l == label]
    label_embeddings[label] = X_train_embed[indices]

# Compute prototype (mean embedding)
label_prototypes = {}

for label, embeds in label_embeddings.items():
    label_prototypes[label] = np.mean(embeds, axis=0)

In [ ]:
labels = list(label_prototypes.keys())

prototype_matrix = np.vstack([label_prototypes[label] for label in labels])


def predict_cosine(X):

    sims = cosine_similarity(X, prototype_matrix)

    pred_indices = np.argmax(sims, axis=1)

    preds = [labels[i] for i in pred_indices]

    return preds

In [20]:
pred_seen = predict_cosine(X_test_seen_embed)

acc_seen = accuracy_score(test_labels_seen, pred_seen)

print("SBERT + Cosine Similarity")
print("Seen Accuracy:", acc_seen)

SBERT + Cosine Similarity
Seen Accuracy: 0.698780487804878


In [21]:
pred_unseen = predict_cosine(X_test_unseen_embed)

acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("Unseen Accuracy:", acc_unseen)

Unseen Accuracy: 0.7771010962241169


## Method 5

**Fine-tuning a Pretrained Transformer**

In this method, we directly fine-tune a pretrained transformer model for task classification. 
Specifically, we use **DistilBERT** (`distilbert-base-uncased`), a lightweight variant of BERT 
that retains most of BERT's language understanding capability while being significantly more 
computationally efficient. DistilBERT is widely used for text classification tasks due to its 
strong performance and reduced model size.

Given an instruction text $d$, we first tokenize the input and feed it into the pretrained 
transformer encoder. The encoder produces a contextual representation:

$$
\mathbf{h} = f_{\theta}(d)
$$

where $f_{\theta}$ denotes the DistilBERT encoder with parameters $\theta$, and 
$\mathbf{h}$ represents the contextual embedding of the input sequence.

The embedding is then passed through a classification layer to produce class probabilities:

$$
\mathbf{y} = \text{softmax}(W\mathbf{h} + b)
$$

where $W$ and $b$ are learnable parameters of the classification head.

Unlike previous methods where the encoder is used as a fixed feature extractor, in this 
approach we **fine-tune the entire model**, including both the transformer encoder and the 
classification layer.

The model parameters are optimized by minimizing the cross-entropy loss:

$$
\mathcal{L} = - \sum_{i=1}^{C} y_i \log(\hat{y}_i)
$$

where $C$ is the number of task classes, $y_i$ is the ground-truth label, and $\hat{y}_i$ 
is the predicted probability for class $i$.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import numpy as np

label_set = sorted(list(set(train_labels)))

label2id = {label: i for i, label in enumerate(label_set)}
id2label = {i: label for label, i in label2id.items()}

y_train = [label2id[l] for l in train_labels]
y_seen = [label2id[l] for l in test_labels_seen]
y_unseen = [label2id[l] for l in test_labels_unseen]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=64)

seen_enc = tokenizer(test_texts_seen, truncation=True, padding=True, max_length=64)

unseen_enc = tokenizer(test_texts_unseen, truncation=True, padding=True, max_length=64)

In [ ]:
import torch


class AlfredDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [25]:
train_dataset = AlfredDataset(train_enc, y_train)

seen_dataset = AlfredDataset(seen_enc, y_seen)

unseen_dataset = AlfredDataset(unseen_enc, y_unseen)

In [26]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(label_set)
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="no",
)

In [ ]:
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset)

In [29]:
trainer.train()
from sklearn.metrics import accuracy_score

pred_seen = trainer.predict(seen_dataset)

y_pred_seen = np.argmax(pred_seen.predictions, axis=1)

acc_seen = accuracy_score(y_seen, y_pred_seen)

print("Fine-tuned Transformer")
print("Seen Accuracy:", acc_seen)

Step,Training Loss
50,1.732558
100,0.976114
150,0.536733
200,0.307676
250,0.298667
300,0.221561
350,0.251917
400,0.198442
450,0.249231
500,0.218894


Fine-tuned Transformer
Seen Accuracy: 0.947560975609756


In [30]:
pred_unseen = trainer.predict(unseen_dataset)

y_pred_unseen = np.argmax(pred_unseen.predictions, axis=1)

acc_unseen = accuracy_score(y_unseen, y_pred_unseen)

print("Unseen Accuracy:", acc_unseen)

Unseen Accuracy: 0.9634591961023142
